# SPAR: Training Pipeline (Colab)

**SPAR** (Sensor-Policy with Adaptive Resource allocation) is a safe reinforcement learning framework that treats sensor selection as part of the policy. At each timestep the agent chooses *which* observation modalities to activate alongside the environment action, subject to a per-step sensor-cost budget enforced as a Constrained Markov Decision Process (CMDP). The Lagrangian multiplier λ adapts online to keep cumulative cost at or below the budget. This notebook trains SPAR on `highway-env` (no MuJoCo required) and visualises reward, cost, and per-sensor activation rates.

## Phase 1: Environment Setup

Colab's default runtime has no display server, which causes `highway-env` to crash when it tries to render. This cell:
- Detects whether a GPU is available (T4 will be used if the runtime was set to GPU)
- Installs `xvfb` (a virtual framebuffer) and starts a headless display so `highway-env` can render without a monitor

In [ ]:
# GPU detection
import torch
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Virtual display (guard for highway-env headless rendering)
import subprocess
subprocess.run(['apt-get', 'install', '-y', '-q', 'xvfb', 'python3-opengl'], check=False)
!pip install -q pyvirtualdisplay
from pyvirtualdisplay import Display
Display(visible=0, size=(1400, 900)).start()

In [ ]:
REPO_URL  = "https://github.com/OfekGlick/SPAR.git"
REPO_ROOT = "/content/SPAR"

import os
if not os.path.exists(REPO_ROOT):
    os.system(f"git clone {REPO_URL} {REPO_ROOT}")
os.chdir(REPO_ROOT)
os.system("git pull")  # pull latest fixes

# Pin numpy to 1.x before anything else — avoids ABI mismatch with Colab's pre-built packages
os.system("pip install -q 'numpy<2.0'")

# Install dependencies: highway-env (editable), then requirements, then root package
os.system(f"pip install -q -e {REPO_ROOT}/base_envs/highway_env")
os.system(f"pip install -q -r {REPO_ROOT}/requirements.txt")
os.system(f"pip install -q -e {REPO_ROOT}")
print("Install complete — RESTART RUNTIME now (Runtime > Restart runtime), then continue.")

> **Important:** After the cell above finishes, go to **Runtime → Restart runtime**, then continue running from the next cell. This ensures newly installed packages are available.

## Phase 2: Imports

Loads the SPAR codebase into Python's module path and imports all required components:
- **`spar_envs.budget_aware_highway`** — registers the custom `budget-aware-highway-fast-v0` environment ID with OmniSafe's registry
- **`omnisafe`** — the safe RL training framework (PPOLag, CPO, etc.)
- **`configs.highway_config`** — default hyperparameter dict (`CUSTOM_CFGS`) for the highway experiment
- **`training_utils`** — helper that wraps OmniSafe's agent with sample-efficiency tracking and periodic evaluation

In [ ]:
import sys, os
REPO_ROOT = "/content/SPAR"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

import spar_envs.budget_aware_highway   # registers env IDs with OmniSafe
import omnisafe, torch, pandas as pd, matplotlib.pyplot as plt
from configs.highway_config import CUSTOM_CFGS
from utils.training_utils import train_agent_base
print("Imports OK")

## Phase 3: Configuration

Sets all experiment hyperparameters before training begins.

| Parameter | Value | Description |
|---|---|---|
| `ENV_ID` | `budget-aware-highway-fast-v0` | Highway environment with sensor budget constraint |
| `ALGO` | `PPOLag` | Lagrangian PPO — adapts λ to keep cost ≤ budget |
| `BUDGET` | `2.0` | Max sensor-cost per step (4 sensors each cost 1.0) |
| `TOTAL_STEPS` | `20480` | Total environment steps for training |
| `steps_per_epoch` | `1024` | Steps collected before each policy update |

**Key flags:**
- `use_cost=True` — enables the safety constraint (CMDP mode); set to `False` for an unconstrained PPO baseline
- `use_all_obs=False` — SPAR mode where the agent selects sensors; `True` gives access to all sensors at every step
- `sd_regulizer=False` — disables the sensor-dropout regulariser (can be enabled for robustness experiments)

In [ ]:
import copy

ENV_ID           = "budget-aware-highway-fast-v0"
ALGO             = "PPOLag"      # PPOLag = safe; PPO = unconstrained baseline
BUDGET           = 50.0          # sensor cost limit per trajectory (50% of max: 4 sensors × 1.0 × 25 steps = 100)
TOTAL_STEPS      = 5000          # smoke test; full training: 409600
SEED             = 42

custom_cfgs = copy.deepcopy(CUSTOM_CFGS)
custom_cfgs['logger_cfgs']['use_wandb']        = False
custom_cfgs['logger_cfgs']['use_tensorboard']  = False
custom_cfgs['logger_cfgs']['log_dir']          = '/content/runs'
custom_cfgs['train_cfgs']['total_steps']       = TOTAL_STEPS
custom_cfgs['train_cfgs']['device']            = 'cuda:0' if torch.cuda.is_available() else 'cpu'
custom_cfgs['algo_cfgs']['steps_per_epoch']    = 250    # full training: 8192
custom_cfgs['env_cfgs']['max_episode_steps']   = 25     # ÷10; full training: 250
custom_cfgs['env_cfgs']['seed']                = SEED
custom_cfgs['env_cfgs']['use_all_obs']         = False  # False = SPAR; True = all-sensors baseline
custom_cfgs['algo_cfgs']['use_cost']           = True
custom_cfgs['lagrange_cfgs']                   = {'cost_limit': BUDGET}
custom_cfgs['algo_cfgs']['sd_regulizer']       = False
custom_cfgs['model_cfgs']['actor_type']        = 'auto'

print(f"Will run {TOTAL_STEPS // 250} epochs ({TOTAL_STEPS} steps) on {custom_cfgs['train_cfgs']['device']}")

## Phase 4: Training

Runs the full training loop via `train_agent_base`, which:
1. Instantiates an OmniSafe `Agent` with the config above
2. Wraps it in a `SampleEfficiencyTracker` that evaluates the policy every 5% of total steps
3. Calls `agent.learn()` — PPOLag collects `steps_per_epoch` environment steps, updates the policy, and adjusts the Lagrange multiplier λ to enforce the cost budget
4. Runs a final evaluation (`eval_num_episodes=3`) after training completes

Progress is logged to `/content/runs/` as a CSV. Training time depends on hardware — expect ~2–5 min on a T4 GPU for this smoke-test configuration.

In [ ]:
import time
t0 = time.time()

train_agent_base(ALGO, ENV_ID, custom_cfgs, eval_num_episodes=1)  # ÷10; full training: 50

print(f"Done in {(time.time()-t0)/60:.1f} min")

## Phase 5: Load Results

Locates the most recently written `progress.csv` under `/content/runs/` and loads it into a DataFrame. Each row is one epoch and includes columns such as:
- `TotalEnvSteps` — cumulative environment steps at that epoch
- `Metrics/EpRet` — mean episode return
- `Metrics/EpCost` — mean episode cost (sensor activations × cost per sensor)
- `Metrics/LagrangeMultiplier` — current value of λ (PPOLag only)

The tail printout lets you quickly verify that training produced logged data before plotting.

In [ ]:
import glob

progress_files = sorted(glob.glob('/content/runs/**/*.csv', recursive=True), key=os.path.getmtime)
assert progress_files, "No progress.csv found — check training completed without errors"
progress_csv = progress_files[-1]
print("Progress file:", progress_csv)

df = pd.read_csv(progress_csv)
print(f"{len(df)} epochs logged")
print(df[['TotalEnvSteps', 'Metrics/EpRet', 'Metrics/EpCost']].tail())

## Phase 6: Training Curves

Plots three panels side-by-side against total environment steps:
- **Episode Return** — cumulative reward per episode; should increase as the policy improves
- **Episode Cost** — cumulative sensor cost per episode, with a dashed horizontal line at the budget limit (`BUDGET × max_episode_steps`); a well-trained PPOLag agent should converge just below this line
- **Lagrange Multiplier λ** — the adaptive penalty that enforces the cost constraint; rises when cost exceeds the budget and falls when it is under

The figure is saved to `/content/training_curves.png`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# EpRet
axes[0].plot(df['TotalEnvSteps'], df['Metrics/EpRet'], '#2E86AB', lw=2)
axes[0].set(title='Episode Return', xlabel='Steps', ylabel='EpRet'); axes[0].grid(alpha=0.3)

# EpCost + budget line (BUDGET is per-trajectory, same units as Metrics/EpCost)
axes[1].plot(df['TotalEnvSteps'], df['Metrics/EpCost'], '#C73E1D', lw=2, label='EpCost')
axes[1].axhline(BUDGET, color='k', ls='--', label=f'Budget={BUDGET:.0f}')
axes[1].set(title='Episode Cost', xlabel='Steps', ylabel='EpCost'); axes[1].legend(); axes[1].grid(alpha=0.3)

# Lagrange multiplier (PPOLag only)
lag_col = next((c for c in df.columns if 'LagrangeMultiplier' in c and '/' not in c.replace('Metrics/LagrangeMultiplier','')), None)
if lag_col:
    axes[2].plot(df['TotalEnvSteps'], df[lag_col], '#6A994E', lw=2)
    axes[2].set(title='Lagrange Multiplier \u03bb', xlabel='Steps'); axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'N/A\n(PPO baseline)', ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title('Lagrange Multiplier \u03bb')

plt.suptitle(f'{ALGO} on {ENV_ID} \u2014 budget={BUDGET}', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 7: Per-Sensor Activation Rates

Shows how often the agent activates each of the four observation modalities over the course of training:

| Sensor | Description |
|---|---|
| **Kinematics** | Ego-vehicle speed, heading, and neighbouring vehicle positions |
| **LidarObservation** | Radial distance scans around the ego vehicle |
| **OccupancyGrid** | Binary occupancy map of the surrounding grid |
| **TimeToCollision** | Predicted time-to-collision for each lane |

The dashed line marks the uniform allocation baseline (equal budget split across all sensors). SPAR is expected to learn a non-uniform allocation — favouring cheaper or more informative sensors — while staying within the budget.

The figure is saved to `/content/sensor_activations.png`. This cell is a no-op when `use_all_obs=True` (all-sensors baseline).

In [ ]:
sensor_cols = sorted([c for c in df.columns if 'EpActivationSensor' in c])
modality_names = ['Kinematics', 'LidarObservation', 'OccupancyGrid', 'TimeToCollision']
colors = ['#2E86AB', '#F18F01', '#C73E1D', '#6A994E']

if sensor_cols:
    fig, ax = plt.subplots(figsize=(9, 4))
    for i, col in enumerate(sensor_cols):
        idx = int(col.split('_')[-1])
        ax.plot(df['TotalEnvSteps'], df[col], color=colors[i], lw=2,
                label=modality_names[idx] if idx < 4 else f"Sensor_{idx}")
    ax.axhline(BUDGET / len(sensor_cols), color='k', ls='--', lw=1, label='Uniform budget')
    ax.set(ylim=(0, 1.05), title='Per-Sensor Activation Rate', xlabel='Steps', ylabel='Rate')
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/sensor_activations.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nFinal epoch activation rates:")
    for i, col in enumerate(sensor_cols):
        print(f"  {modality_names[int(col.split('_')[-1])]}: {df[col].iloc[-1]:.3f}")
else:
    print("Sensor activation columns not found (expected when use_all_obs=True)")

## Scaling Up

Full-scale experiment (single run):
```bash
python run_spar_highway.py --algo PPOLag --env-id budget-aware-highway-fast-v0 \
    --budget 2.0 --total-steps 409600 --seed 42
```

All environments + budgets on a cluster (generates Slurm jobs from `configs/highway_config.py`):
```bash
python launch_highway.py --submit
```

Robosuite (requires MuJoCo ≥ 3.3.0):
```bash
cd base_envs/robosuite && pip install -e .
python run_spar_robosuite.py --algo PPOLag --env-id budget-aware-Lift --budget 3.0
```